# FMP 재무 데이터 조회 노트북

**Financial Modeling Prep API** 를 이용해 ticker 별 재무 데이터를 조회합니다.

---
| 셀 | 단계 |
|----|------|
| 1  | 환경 설정 & 경로 자동 감지 |
| 2  | 설정 상수 (API Key / 파라미터) |
| 3  | FMP API 호출 함수 정의 |
| 4  | **조회 가능한 재무 항목명 확인** |
| 5  | **최근 N분기 재무 데이터 조회** |
| 6  | 여러 항목 동시 조회 |

## Cell 1 · 환경 설정 & 경로 자동 감지

In [1]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# ── 후보 프로젝트 루트 (노트북 / 데스크탑) ─────────────────────
_CANDIDATE_ROOTS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast",         # 노트북
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy",  # 데스크탑
]

def _setup_path() -> str:
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd()
    for p in [start] + list(start.parents):
        if (p / "DATA").is_dir():
            root = str(p)
            if root not in sys.path:
                sys.path.insert(0, root)
            print(f"[PATH] root 자동 감지 : {root}")
            return root
    for candidate in _CANDIDATE_ROOTS:
        if os.path.isdir(candidate) and os.path.isdir(os.path.join(candidate, "DATA")):
            if candidate not in sys.path:
                sys.path.insert(0, candidate)
            print(f"[PATH] root 후보 경로 : {candidate}")
            return candidate
    raise EnvironmentError("DATA 폴더를 찾을 수 없습니다.")

_ROOT = _setup_path()
print(f"[확인] 프로젝트 루트 : {_ROOT}")


[PATH] root 자동 감지 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[확인] 프로젝트 루트 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy


## Cell 2 · 설정 상수

**여기만 수정하세요**

In [2]:
import time
import requests
import pandas as pd
from typing import Optional, List
from IPython.display import display
from DATA.config import get_db_info, get_engine
from DATA.us_target_ticker_list_2000 import ticker_list as DEFAULT_TICKER_LIST

# ══════════════════════════════════════════════════════
#  ★ 여기만 수정하세요 ★
# ══════════════════════════════════════════════════════
API_KEY      = "hT0gAk87j9xZx4PlBApvBqfVL5IahvgV"

TEST_TICKER  = "MU"   # 조회할 ticker
N_QUARTERS   = 8        # 최근 N분기 (default 8)
PERIOD       = "quarter" # "quarter" 또는 "annual"
# ══════════════════════════════════════════════════════

FMP_BASE     = "https://financialmodelingprep.com/api/v3"
MAX_RETRY    = 3
SLEEP_SEC    = 0.35

print(f"[설정] API_KEY    = {API_KEY[:8]}...")
print(f"[설정] TEST_TICKER= {TEST_TICKER}")
print(f"[설정] N_QUARTERS = {N_QUARTERS}")
print(f"[설정] PERIOD     = {PERIOD}")


[설정] API_KEY    = hT0gAk87...
[설정] TEST_TICKER= MU
[설정] N_QUARTERS = 8
[설정] PERIOD     = quarter


## Cell 3 · FMP API 호출 함수 정의

In [3]:
def _fmp_get(endpoint: str, params: dict = None) -> list:
    """
    FMP API GET 요청 (재시도 포함).
    endpoint 예: '/income-statement/AAPL'
    """
    url = FMP_BASE + endpoint
    _params = {"apikey": API_KEY}
    if params:
        _params.update(params)

    for k in range(MAX_RETRY):
        try:
            r = requests.get(url, params=_params, timeout=30)
            if r.status_code == 429:          # Rate limit
                time.sleep(1.5 + k)
                continue
            r.raise_for_status()
            data = r.json()
            # FMP 는 오류 시 dict {"Error Message": "..."} 반환
            if isinstance(data, dict) and "Error Message" in data:
                raise ValueError(data["Error Message"])
            return data if isinstance(data, list) else []
        except Exception as e:
            if k == MAX_RETRY - 1:
                raise
            time.sleep(SLEEP_SEC + k * 0.5)
    return []


def fetch_income_statement(
    ticker: str,
    n: int    = 8,
    period: str = "quarter",
) -> pd.DataFrame:
    """
    손익계산서 (Income Statement) 조회.
    revenue, grossProfit, operatingIncome, netIncome 등 포함.
    """
    data = _fmp_get(f"/income-statement/{ticker}",
                    {"period": period, "limit": n})
    if not data:
        return pd.DataFrame()
    df = pd.DataFrame(data)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    return df.sort_values("date").reset_index(drop=True)


def fetch_balance_sheet(
    ticker: str,
    n: int      = 8,
    period: str = "quarter",
) -> pd.DataFrame:
    """
    재무상태표 (Balance Sheet) 조회.
    totalAssets, totalDebt, cashAndEquivalents 등 포함.
    """
    data = _fmp_get(f"/balance-sheet-statement/{ticker}",
                    {"period": period, "limit": n})
    if not data:
        return pd.DataFrame()
    df = pd.DataFrame(data)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    return df.sort_values("date").reset_index(drop=True)


def fetch_cash_flow(
    ticker: str,
    n: int      = 8,
    period: str = "quarter",
) -> pd.DataFrame:
    """
    현금흐름표 (Cash Flow Statement) 조회.
    operatingCashFlow, freeCashFlow, capitalExpenditure 등 포함.
    """
    data = _fmp_get(f"/cash-flow-statement/{ticker}",
                    {"period": period, "limit": n})
    if not data:
        return pd.DataFrame()
    df = pd.DataFrame(data)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    return df.sort_values("date").reset_index(drop=True)


def fetch_key_metrics(
    ticker: str,
    n: int      = 8,
    period: str = "quarter",
) -> pd.DataFrame:
    """
    핵심 지표 (Key Metrics) 조회.
    peRatio, pbRatio, evToEbitda, roe, roa 등 포함.
    """
    data = _fmp_get(f"/key-metrics/{ticker}",
                    {"period": period, "limit": n})
    if not data:
        return pd.DataFrame()
    df = pd.DataFrame(data)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    return df.sort_values("date").reset_index(drop=True)


def fetch_financial_ratios(
    ticker: str,
    n: int      = 8,
    period: str = "quarter",
) -> pd.DataFrame:
    """
    재무 비율 (Financial Ratios) 조회.
    currentRatio, debtRatios, profitabilityRatios 등 포함.
    """
    data = _fmp_get(f"/ratios/{ticker}",
                    {"period": period, "limit": n})
    if not data:
        return pd.DataFrame()
    df = pd.DataFrame(data)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    return df.sort_values("date").reset_index(drop=True)


print("[OK] FMP API 함수 정의 완료")
print("  fetch_income_statement  : 손익계산서")
print("  fetch_balance_sheet     : 재무상태표")
print("  fetch_cash_flow         : 현금흐름표")
print("  fetch_key_metrics       : 핵심 지표 (PER, PBR, ROE 등)")
print("  fetch_financial_ratios  : 재무 비율")


[OK] FMP API 함수 정의 완료
  fetch_income_statement  : 손익계산서
  fetch_balance_sheet     : 재무상태표
  fetch_cash_flow         : 현금흐름표
  fetch_key_metrics       : 핵심 지표 (PER, PBR, ROE 등)
  fetch_financial_ratios  : 재무 비율


## Cell 4 · 조회 가능한 재무 항목명 확인

각 재무제표별로 **어떤 컬럼(항목)을 조회할 수 있는지** 한눈에 확인합니다.

In [4]:
# ── 각 재무제표별 항목명 조회 ────────────────────────────────
# TEST_TICKER 의 최근 1개 분기 데이터를 가져와서 컬럼명만 출력

STATEMENTS = {
    "손익계산서 (Income Statement)" : fetch_income_statement,
    "재무상태표 (Balance Sheet)"    : fetch_balance_sheet,
    "현금흐름표 (Cash Flow)"        : fetch_cash_flow,
    "핵심지표   (Key Metrics)"      : fetch_key_metrics,
    "재무비율   (Financial Ratios)" : fetch_financial_ratios,
}

# 메타 컬럼 (항목명이 아닌 식별 컬럼)
META_COLS = {
    "date", "symbol", "reportedCurrency", "cik",
    "fillingDate", "acceptedDate", "calendarYear",
    "period", "link", "finalLink",
}

all_fields = {}  # {재무제표명: [항목명 리스트]}

for stmt_name, fn in STATEMENTS.items():
    try:
        df = fn(TEST_TICKER, n=1, period=PERIOD)
        if df.empty:
            print(f"  [{stmt_name}] 데이터 없음")
            continue
        # 메타 컬럼 제외한 실제 재무 항목만 추출
        fields = [c for c in df.columns if c not in META_COLS]
        all_fields[stmt_name] = fields
    except Exception as e:
        print(f"  [{stmt_name}] 오류: {e}")
        all_fields[stmt_name] = []

# ── 출력 ─────────────────────────────────────────────────────
print(f"[{TEST_TICKER}] 조회 가능한 재무 항목 목록")
print("=" * 70)
for stmt_name, fields in all_fields.items():
    print(f"\n■ {stmt_name}  ({len(fields)}개 항목)")
    print("-" * 50)
    for i, f in enumerate(fields, 1):
        print(f"  {i:>3}. {f}")

# ── DataFrame 으로도 확인 ────────────────────────────────────
print("\n" + "=" * 70)
print("[전체 항목 DataFrame]")
rows = []
for stmt_name, fields in all_fields.items():
    for f in fields:
        rows.append({"재무제표": stmt_name, "항목명(영문)": f})
fields_df = pd.DataFrame(rows)
display(fields_df)
print(f"\n총 {len(fields_df)}개 항목 조회 가능")


[MU] 조회 가능한 재무 항목 목록

■ 손익계산서 (Income Statement)  (28개 항목)
--------------------------------------------------
    1. revenue
    2. costOfRevenue
    3. grossProfit
    4. grossProfitRatio
    5. researchAndDevelopmentExpenses
    6. generalAndAdministrativeExpenses
    7. sellingAndMarketingExpenses
    8. sellingGeneralAndAdministrativeExpenses
    9. otherExpenses
   10. operatingExpenses
   11. costAndExpenses
   12. interestIncome
   13. interestExpense
   14. depreciationAndAmortization
   15. ebitda
   16. ebitdaratio
   17. operatingIncome
   18. operatingIncomeRatio
   19. totalOtherIncomeExpensesNet
   20. incomeBeforeTax
   21. incomeBeforeTaxRatio
   22. incomeTaxExpense
   23. netIncome
   24. netIncomeRatio
   25. eps
   26. epsdiluted
   27. weightedAverageShsOut
   28. weightedAverageShsOutDil

■ 재무상태표 (Balance Sheet)  (44개 항목)
--------------------------------------------------
    1. cashAndCashEquivalents
    2. shortTermInvestments
    3. cashAndShortTermInvestments


,재무제표,항목명(영문)
0,손익계산서 (Income Statement),revenue
1,손익계산서 (Income Statement),costOfRevenue
2,손익계산서 (Income Statement),grossProfit
3,손익계산서 (Income Statement),grossProfitRatio
4,손익계산서 (Income Statement),researchAndDevelopmentExpenses
...,...,...
208,재무비율 (Financial Ratios),priceEarningsToGrowthRatio
209,재무비율 (Financial Ratios),priceSalesRatio
210,재무비율 (Financial Ratios),dividendYield
211,재무비율 (Financial Ratios),enterpriseValueMultiple



총 213개 항목 조회 가능


## Cell 5 · 최근 N분기 재무 데이터 조회

**`QUERY_TICKER`** 와 **`QUERY_ITEMS`** 를 지정하면 해당 항목만 추출합니다.

Cell 4 에서 확인한 항목명을 `QUERY_ITEMS` 에 넣으세요.

In [7]:
# ══════════════════════════════════════════════════════
#  ★ 여기를 수정하세요 ★
# ══════════════════════════════════════════════════════
QUERY_TICKER = TEST_TICKER       # 조회할 ticker
QUERY_N      = 80           # 최근 N분기
QUERY_PERIOD = "quarter"   # "quarter" / "annual"

# 조회할 항목명 — Cell 4 에서 확인한 영문 컬럼명 사용
# 예: 손익계산서 항목
QUERY_ITEMS = [
    "revenue",              # 매출
    "grossProfit",          # 매출총이익
    "operatingIncome",      # 영업이익
    "netIncome",            # 순이익
    "ebitda",               # EBITDA
    "eps",                  # 주당순이익
    "grossProfitRatio",     # 매출총이익률
    "operatingIncomeRatio", # 영업이익률
    "netIncomeRatio",       # 순이익률
]
# ══════════════════════════════════════════════════════


def fetch_selected_items(
    ticker: str,
    items: List[str],
    n: int       = 8,
    period: str  = "quarter",
) -> pd.DataFrame:
    """
    지정한 항목들을 모든 재무제표에서 검색해 통합 반환합니다.

    Parameters
    ----------
    ticker : 종목 코드
    items  : 조회할 항목명 리스트 (영문 컬럼명)
    n      : 최근 N분기
    period : 'quarter' / 'annual'

    Returns
    -------
    DataFrame  index=date, columns=items
    """
    fetchers = [
        fetch_income_statement,
        fetch_balance_sheet,
        fetch_cash_flow,
        fetch_key_metrics,
        fetch_financial_ratios,
    ]

    result = None
    found  = set()

    for fn in fetchers:
        remaining = [it for it in items if it not in found]
        if not remaining:
            break

        try:
            df = fn(ticker, n=n, period=period)
        except Exception as e:
            print(f"  [WARN] {fn.__name__}: {e}")
            continue

        if df.empty:
            continue

        # 이번 재무제표에서 찾은 항목
        hit_cols = [c for c in remaining if c in df.columns]
        if not hit_cols:
            continue

        sub = df[["date"] + hit_cols].set_index("date")

        if result is None:
            result = sub
        else:
            result = result.join(sub, how="outer")

        found.update(hit_cols)
        time.sleep(SLEEP_SEC)

    # 못 찾은 항목 경고
    not_found = [it for it in items if it not in found]
    if not_found:
        print(f"  [WARN] 찾을 수 없는 항목: {not_found}")
        print(f"         Cell 4 에서 정확한 항목명을 확인하세요.")

    if result is None:
        return pd.DataFrame()

    return result.sort_index()


# ── 실행 & 출력 ──────────────────────────────────────────────
print(f"[조회] {QUERY_TICKER}  최근 {QUERY_N}분기  ({QUERY_PERIOD})")
print(f"       항목: {QUERY_ITEMS}")
print("-" * 70)

result_df = fetch_selected_items(
    ticker = QUERY_TICKER,
    items  = QUERY_ITEMS,
    n      = QUERY_N,
    period = QUERY_PERIOD,
)

if result_df.empty:
    print("[결과 없음]")
else:
    print(f"\n[결과] {QUERY_TICKER}  {len(result_df)}개 분기  ×  {len(result_df.columns)}개 항목")

    # 숫자 포맷 (큰 수는 억 단위로 표시)
    def _fmt(x):
        if pd.isna(x):
            return ""
        if abs(x) >= 1_000_000_000:
            return f"{x/1_000_000_000:.2f}B"
        if abs(x) >= 1_000_000:
            return f"{x/1_000_000:.2f}M"
        return f"{x:.4f}"

    disp = result_df.copy()
    disp.index = disp.index.strftime("%Y-%m-%d")
    display(disp.applymap(_fmt))


[조회] MU  최근 80분기  (quarter)
       항목: ['revenue', 'grossProfit', 'operatingIncome', 'netIncome', 'ebitda', 'eps', 'grossProfitRatio', 'operatingIncomeRatio', 'netIncomeRatio']
----------------------------------------------------------------------

[결과] MU  80개 분기  ×  9개 항목


,revenue,grossProfit,operatingIncome,netIncome,ebitda,eps,grossProfitRatio,operatingIncomeRatio,netIncomeRatio
date,,,,,,,,,
2006-08-31,1.37B,323.80M,53.00M,63.70M,396.80M,0.0885,0.2359,0.0386,0.0464
2006-11-30,1.53B,442.00M,110.00M,115.00M,490.00M,0.1500,0.2889,0.0719,0.0752
2007-03-01,1.43B,357.00M,-34.00M,-52.00M,386.00M,-0.0676,0.2502,-0.0238,-0.0364
2007-05-31,1.29B,106.00M,-195.00M,-225.00M,249.00M,-0.2900,0.0819,-0.1507,-0.1739
2007-10-12,1.44B,173.00M,-161.00M,-158.00M,332.00M,-0.2100,0.1204,-0.1120,-0.1100
...,...,...,...,...,...,...,...,...,...
2025-05-29,9.30B,3.51B,2.17B,1.89B,4.33B,1.6900,0.3772,0.2332,0.2027
2025-08-28,11.31B,5.05B,3.75B,3.20B,5.95B,2.8600,0.4467,0.3318,0.2829
2025-11-27,13.64B,7.65B,6.14B,5.24B,8.35B,4.6600,0.5604,0.4498,0.3841


In [9]:
result_df.to_csv(rf"C:\Users\82108\OneDrive\INVESTMENT\미국주식\raw_data\{TEST_TICKER}_fs_data.csv")

## Cell 6 · 단일 재무제표 전체 조회 (원본 숫자)

특정 재무제표 전체를 원본 숫자 그대로 확인합니다.

In [6]:
# ══════════════════════════════════════════════════════
#  ★ 여기를 수정하세요 ★
# ══════════════════════════════════════════════════════
RAW_TICKER    = "AAPL"
RAW_N         = 8
RAW_PERIOD    = "quarter"
RAW_STATEMENT = "income"   # income / balance / cashflow / metrics / ratios
# ══════════════════════════════════════════════════════

_FN_MAP = {
    "income"   : fetch_income_statement,
    "balance"  : fetch_balance_sheet,
    "cashflow" : fetch_cash_flow,
    "metrics"  : fetch_key_metrics,
    "ratios"   : fetch_financial_ratios,
}

_NAME_MAP = {
    "income"   : "손익계산서",
    "balance"  : "재무상태표",
    "cashflow" : "현금흐름표",
    "metrics"  : "핵심지표",
    "ratios"   : "재무비율",
}

fn = _FN_MAP.get(RAW_STATEMENT)
if fn is None:
    print(f"[오류] RAW_STATEMENT 는 {list(_FN_MAP.keys())} 중 하나여야 합니다.")
else:
    raw_df = fn(RAW_TICKER, n=RAW_N, period=RAW_PERIOD)

    if raw_df.empty:
        print("[결과 없음]")
    else:
        # 메타 컬럼 제외
        META_COLS = {
            "symbol", "reportedCurrency", "cik",
            "fillingDate", "acceptedDate", "link", "finalLink"
        }
        show_cols = [c for c in raw_df.columns if c not in META_COLS]
        raw_df = raw_df[show_cols].set_index("date")
        raw_df.index = raw_df.index.strftime("%Y-%m-%d")

        print(f"[{RAW_TICKER}] {_NAME_MAP[RAW_STATEMENT]}  최근 {len(raw_df)}분기")
        print(f"항목 수: {len(raw_df.columns)}개")
        display(raw_df.T)   # 행=항목, 열=분기 (읽기 편하게 전치)


[AAPL] 손익계산서  최근 8분기
항목 수: 30개


date,2024-09-28,2024-12-28,2025-03-29,2025-06-28,2025-09-27,2025-12-27,2026-03-28,2026-06-27
calendarYear,2024,2025,2025,2025,2025,2026,2026,2026
period,Q4,Q1,Q2,Q3,Q4,Q1,Q2,Q3
revenue,94930000000,124300000000,95359000000,94036000000,102466000000,143756000000,111184000000,109417000000
costOfRevenue,51051000000,66025000000,50492000000,50318000000,54125000000,74525000000,56403000000,54647000000
grossProfit,43879000000,58275000000,44867000000,43718000000,48341000000,69231000000,54781000000,54770000000
grossProfitRatio,0.462225,0.468825,0.470506,0.464907,0.471776,0.481587,0.492706,0.500562
researchAndDevelopmentExpenses,7765000000,8268000000,8550000000,8866000000,8866000000,10887000000,11419000000,11729000000
generalAndAdministrativeExpenses,0,2074000000,0,0,0,2095000000,0,0
sellingAndMarketingExpenses,0,5101000000,0,0,0,5397000000,0,0
sellingGeneralAndAdministrativeExpenses,6523000000,7175000000,6728000000,6650000000,7048000000,7492000000,7477000000,7346000000
